In [ ]:
%pip install plotly numpy pandas catboost scipy scikit-learn shap statsmodels

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.figure_factory as ff

from catboost import CatBoostRegressor
import scipy.stats as stats
from scipy.stats import spearmanr
from scipy.stats import f_oneway
from scipy.stats import chi2_contingency
from statsmodels.stats.weightstats import ztest as ztest_lib
from scipy.stats import fisher_exact
from scipy.stats import kendalltau
from sklearn.model_selection import train_test_split
import shap

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bobrnebobr/Stats-Lab/refs/heads/main/data/combined_cycle_power_plant.csv")

## Исследование данных

In [ ]:
df.shape

In [ ]:
ff.create_distplot([df["pm2.5"].dropna()], ["pm2.5"])

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
df = df.dropna()

## Статистические тесты

#### Матрица корреляции

In [ ]:
num_cols = ["DEWP","TEMP","PRES","Iws","Is","Ir", "sensor_noise", "traffic_index"]
px.imshow(df[num_cols + ["pm2.5"]].corr(), text_auto=True,
          title="Матрица корреляции для фичей")


#### ANOVA для cbwd

In [ ]:
fig_wind = px.box(df, x="cbwd", y="pm2.5",
                  color="cbwd",
                  title="Распределение PM2.5 в зависимости от направления ветра",
                  labels={"cbwd": "Направление ветра", "pm2.5": "Концентрация PM2.5"})
fig_wind.show()

In [ ]:
groups = [group["pm2.5"].values for _, group in df.groupby("cbwd")]

f_stat, p_val = f_oneway(*groups)

print("--- ТЕСТ ANOVA ---")
print("F-статистика:", f_stat)
print("p-value:", p_val)

if p_val < 0.05:
    print("Направление ветра является статистически значимым фактором")
else:
    print("Статистически значимых различий не обнаружено")

ANOVA по месту нахождения датчика

In [ ]:
fig_wind = px.box(df, x="sensor_zone", y="pm2.5",
                  color="sensor_zone",
                  title="Распределение PM2.5 в зависимости от направления ветра",
                  labels={"sensor_zone": "Местонахождение датчика", "pm2.5": "Концентрация PM2.5"})
fig_wind.show()

In [ ]:
groups = [group["pm2.5"].values for _, group in df.groupby("sensor_zone")]

f_stat, p_val = f_oneway(*groups)

print("--- ТЕСТ ANOVA ---")
print("F-статистика:", f_stat)
print("p-value:", p_val)

if p_val < 0.05:
    print("Направление ветра является статистически значимым фактором")
else:
    print("Статистически значимых различий не обнаружено")

#### ANOVA по часам

In [ ]:
hourly_mean = df.groupby('hour')['pm2.5'].mean().reset_index()
fig_hour = px.line(hourly_mean, x='hour', y='pm2.5', title="Среднее загрязнение по часам", markers=True, line_shape='spline')
fig_hour.show()

In [ ]:
hour_groups = [group["pm2.5"].values for _, group in df.groupby("hour")]
f_hour, p_hour = stats.f_oneway(*hour_groups)
print(f"Час (hour): F-stat={f_hour:.4f}, p-value={p_hour:.10f}")

#### Хи-квадрат для месяцев

In [ ]:
df_temp = df.copy()
df_temp['month_str'] = df_temp['month'].astype(str)

fig_month = px.box(df_temp, x="month_str", y="pm2.5",
                   title="Сезонность: Распределение PM2.5 по месяцам",
                   category_orders={"month_str": [str(i) for i in range(1, 13)]},
                   labels={"month_str": "Месяц", "pm2.5": "Концентрация PM2.5"})
fig_month.show()

In [ ]:
df["pm2_5_bin"] = (df["pm2.5"] > df["pm2.5"].median()).astype(int)


contingency_table = pd.crosstab(df['month'], df['pm2_5_bin'])

print("--- ТАБЛИЦА СОПРЯЖЕННОСТИ (Месяц vs Загрязнение) ---")
print(contingency_table)

In [ ]:
chi2, p_val, dof, expected = chi2_contingency(contingency_table)

print("\n--- ТЕСТ ХИ-КВАДРАТ  ---")
print(f"Статистика Хи-квадрат: {chi2:.4f}")
print(f"p-value: {p_val:.10f}")
print(f"Степени свободы: {dof}")

if p_val < 0.05:
    print("Существует статистически значимая зависимость между месяцем и уровнем загрязнения")
else:
    print("Вывод: Зависимость не обнаружена")

#### Тест Фишера для Temp

In [ ]:
fig_scatter = px.scatter(df.sample(2000), x="TEMP", y="pm2.5",
                         trendline="ols",
                         title="Связь Температуры и PM2.5 (Выборка 2000 точек)",
                         labels={"TEMP": "Температура (°C)", "pm2.5": "Концентрация PM2.5"},
                         opacity=0.5)
fig_scatter.show()

In [ ]:
df['temp_group'] = df['TEMP'].apply(lambda x: 'Холодно' if x <= df['TEMP'].median() else 'Тепло')

fig = px.box(df, x="temp_group", y="pm2.5", color="temp_group",
             title="Сравнение концентрации PM2.5 в холодные и теплые дни",
             labels={"temp_group": "Температурный режим", "pm2.5": "Концентрация PM2.5"})
fig.show()

In [ ]:
df['is_winter'] = df['month'].isin([12, 1, 2]) # зима

threshold = df['pm2.5'].median()
df['is_dirty'] = df['pm2.5'] > 100 # грязно

contingency_table = pd.crosstab(df['is_winter'], df['is_dirty'])

print("--- ТАБЛИЦА СОПРЯЖЕННОСТИ 2x2 ---")
print(contingency_table)

In [ ]:
odds_ratio, p_value = fisher_exact(contingency_table)

print("\n--- ТОЧНЫЙ ТЕСТ ФИШЕРА ---")
print(f"Отношение шансов (Odds Ratio): {odds_ratio:.4f}")
print(f"p-value: {p_value:.10f}")

if p_value < 0.05:
    print("Связь между сезоном (зима) и загрязнением статистически значима")
    print(f"Шанс увидеть грязный воздух зимой в {odds_ratio:.2f} раз выше, чем в другое время")
else:
    print("Вывод: Статистически значимой связи не обнаружено.")

#### Z-тест

In [ ]:
group_cold = df[df['temp_group'] == 'Холодно']['pm2.5']
group_hot = df[df['temp_group'] == 'Тепло']['pm2.5']

z_stat, p_val = ztest_lib(group_cold, group_hot)

print("--- Z-ТЕСТ ---")
print(f"Z-статистика: {z_stat:.4f}")
print(f"p-value: {p_val:.10f}")

if p_val < 0.05:
    print("Разница средних значений PM2.5 в холодные и теплые дни статистически значима")
else:
    print("Статистически значимых различий не обнаружено.")

#### Корреляция Пирсона для DEWP

In [ ]:
def manual_correlation_test(x, y, alpha=0.05):
    n = len(x)

    mu_x = np.mean(x)
    mu_y = np.mean(y)

    numerator = np.sum((x - mu_x) * (y - mu_y))
    denominator = np.sqrt(np.sum((x - mu_x)**2) * np.sum((y - mu_y)**2))

    r = numerator / denominator

    t_stat = r * np.sqrt((n - 2) / (1 - r**2))

    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), n - 2))

    t_crit = stats.t.ppf(1 - alpha/2, n - 2)

    print(f"Коэффициент r: {r:.4f}")
    print(f"Статистика критерия (t): {t_stat:.4f}")
    print(f"p-value: {p_val:.10f}")
    print(f"Критическая область: (-inf, {-t_crit:.4f}] U [{t_crit:.4f}, +inf)")
    print(f"Область принятия: ({-t_crit:.4f}, {t_crit:.4f})")

    return t_stat, p_val

manual_correlation_test(df['DEWP'], df['pm2.5'])

Для индекса трафика

In [ ]:
fig_traffic = px.scatter(df.sample(2000), x="traffic_index", y="pm2.5",
                         trendline="ols",
                         title="Связь Индекса трафика и PM2.5 (Визуальная проверка незначимости)",
                         opacity=0.5)
fig_traffic.show()

In [ ]:
t_stat_tr, p_val_tr = manual_correlation_test(df['traffic_index'], df['pm2.5'])

print(f"\nРезультат для traffic_index:")
if p_val_tr > 0.05:
    print(f"p-value ({p_val_tr:.4f}) > 0.05. Нулевая гипотеза ПРИНИМАЕТСЯ")
    print("traffic_index является статистически НЕЗНАЧИМЫМ фактором")
else:
    print("Фактор неожиданно оказался значимым")

#### T-тест

In [ ]:
df['has_snow'] = df['Is'] > 0
df['has_rain'] = df['Ir'] > 0

In [ ]:
fig_snow = px.box(df, x="has_snow", y="pm2.5", title="Влияние Снега на PM2.5")
fig_snow.show()

In [ ]:
fig_rain = px.box(df, x="has_rain", y="pm2.5", title="Влияние Дождя на PM2.5")
fig_rain.show()

In [ ]:
def manual_ttest_precipitation(group1, group2, feature_name, alpha=0.05):
    n1, n2 = len(group1), len(group2)
    m1, m2 = np.mean(group1), np.mean(group2)
    v1, v2 = np.var(group1, ddof=1), np.var(group2, ddof=1)

    t_stat = (m1 - m2) / np.sqrt(v1/n1 + v2/n2) #формула Уэлча

    df_val = n1 + n2 - 2 # степени свободы

    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df_val))

    t_crit = stats.t.ppf(1 - alpha/2, df_val)

    print(f"--- РУЧНОЙ T-ТЕСТ: Влияние фактора '{feature_name}' ---")
    print(f"Среднее PM2.5 (При осадках): {m1:.2f}")
    print(f"Среднее PM2.5 (Без осадков): {m2:.2f}")
    print(f"Статистика критерия: {t_stat:.4f}")
    print(f"p-value: {p_val:.10f}")
    print(f"Критическая область: (-inf, {-t_crit:.4f}] U [{t_crit:.4f}, +inf)")
    print(f"Область принятия: ({-t_crit:.4f}, {t_crit:.4f})")

    if abs(t_stat) > t_crit:
        print(f"Отклоняем H0. {feature_name} значимо влияет на PM2.5.")
    else:
        print(f"Принимаем H0. {feature_name} не влияет значимо.")
    print("-" * 50)
    return t_stat, p_val

In [ ]:
snow_yes = df[df['Is'] > 0]['pm2.5']
snow_no = df[df['Is'] == 0]['pm2.5']

rain_yes = df[df['Ir'] > 0]['pm2.5']
rain_no = df[df['Ir'] == 0]['pm2.5']

In [ ]:
manual_ttest_precipitation(snow_yes, snow_no, "Снег (Is)")
manual_ttest_precipitation(rain_yes, rain_no, "Дождь (Ir)")

#### Корреляция Спирмена

In [ ]:
fig_pres = px.scatter(df.sample(2000), x="PRES", y="pm2.5", trendline="ols",
                      title="Связь Давления (PRES) и PM2.5", opacity=0.5)
fig_pres.show()

In [ ]:
fig_iws = px.scatter(df.sample(2000), x="Iws", y="pm2.5", trendline="ols",
                     title="Связь Скорости ветра (Iws) и PM2.5", opacity=0.5)
fig_iws.show()

In [ ]:
fig_iws = px.scatter(df.sample(2000), x="sensor_noise", y="pm2.5", trendline="ols",
                     title="Связь шума на сенсорах и PM2.5", opacity=0.5)
fig_iws.show()

In [ ]:
corr_pres, p_pres = spearmanr(df['PRES'], df['pm2.5'])
corr_iws, p_iws = spearmanr(df['Iws'], df['pm2.5'])
corr_noise, p_noise = spearmanr(df['sensor_noise'], df['pm2.5'])

print(f"PRES: Корреляция={corr_pres:.4f}, p-value={p_pres:.10f}")
print(f"Iws:  Корреляция={corr_iws:.4f}, p-value={p_iws:.10f}")
print(f"Noise:  Корреляция={corr_noise:.4f}, p-value={p_noise:.10f}")

#### Критерий Кендалла

In [ ]:
df_sample = df.sample(2000, random_state=42)

fig = px.scatter(df_sample, x="Iws", y="pm2.5", trendline="lowess",
                 title="Кендалл: Связь скорости ветра и PM2.5 (нелинейный тренд)")
fig.show()

In [ ]:
tau, p_val = kendalltau(df_sample['Iws'], df_sample['pm2.5'])

print("--- КРИТЕРИЙ КЕНДАЛЛА (Iws vs PM2.5) ---")
print(f"Коэффициент Тау-Кендалла: {tau:.4f}")
print(f"p-value: {p_val:.10f}")

### Итоги отбора признаков

| Фактор | Примененный тест                | p-value    | Решение | Обоснование |
| :--- |:--------------------------------|:-----------| :--- | :--- |
| **DEWP** | Корреляция Пирсона (**Ручной**) | < 0.05     | **Включить** | Сильная линейная связь |
| **TEMP** | Z-тест                          | < 0.05     | **Включить** | Значимое влияние температуры |
| **PRES** | Корреляция Спирмена             | < 0.05     | **Включить** | Значимая нелинейная связь |
| **Iws** | Критерий Кендалла               | < 0.05     | **Включить** | Ветер очищает воздух |
| **cbwd** | ANOVA                           | < 0.05     | **Включить** | Направление потоков важно |
| **month** | Хи-квадрат                      | < 0.05     | **Включить** | Выраженная сезонность |
| **hour** | ANOVA                           | < 0.05     | **Включить** | Суточный цикл загрязнения |
| **Is / Ir** | T-тест Уэлча (**Ручной**)       | < 0.05     | **Включить** | Осадки снижают PM2.5 |
| **sensor_zone** | ANOVA                           | **0.2808** | **Исключить** | Различия между зонами случайны |
| **sensor_noise**| Корреляция почти 0              | **~ 0**    | **Исключить** | Статистический шум |
| **traffic_index**| Корреляция Пирсона (**Ручной**) | **> 0.05** | **Исключить** | Искусственный шум |

# Обучение модели

In [ ]:
df = pd.read_csv("https://raw.githubusercontent.com/bobrnebobr/Stats-Lab/refs/heads/main/data/combined_cycle_power_plant.csv")
df = df.dropna()
X = df.drop(columns="pm2.5")
y = df["pm2.5"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
X_train.head()

In [ ]:
model = CatBoostRegressor(
    iterations=50_000,
    learning_rate=0.1,
    early_stopping_rounds=100,
    use_best_model=True,
    loss_function='RMSE',
    depth=6,
    cat_features=["cbwd", "year", "month", "day", "sensor_zone"],
    verbose=100,
    random_seed=42,
)

In [ ]:
model.fit(X_train, y_train, eval_set=(X_test, y_test))

In [ ]:
shap.initjs()

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_train)

In [ ]:
shap.plots.beeswarm(shap_values, max_display=20)

In [ ]:
features_to_keep = ["year", "month", "day", "hour", "DEWP", "TEMP", "PRES", "cbwd", "Iws", "Is", "Ir"]

X = df[features_to_keep]
y = df["pm2.5"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, shuffle=True)

In [ ]:
model = CatBoostRegressor(
    iterations=50_000,
    learning_rate=0.1,
    early_stopping_rounds=100,
    use_best_model=True,
    loss_function='RMSE',
    depth=6,
    cat_features=["cbwd", "year", "month", "day"],
    verbose=100,
    random_seed=42,
)

model.fit(X_train, y_train, eval_set=(X_test, y_test))